In [0]:
%sql
USE CATALOG ipl_2024_project;
USE SCHEMA pyspark;

#### Reference diag -->

![image_1781069851421.png](./image_1781069851421.png "image_1781069851421.png")

![image_1781506950931.png](./image_1781506950931.png "image_1781506950931.png")

### Bowling Scorecard (Batsman Level)

#### Information Required
###### 1. Bowling Team Name
###### 2. Bowler Name
###### 3. Overs
###### 4. Runs
###### 5. Wickets
###### 6. Economy
###### 7. Dots

In [0]:
team = spark.read.table("team")
innings = spark.read.table("innings")
player = spark.read.table("player")
score_by_ball = spark.read.table("score_by_ball")

In [0]:
from pyspark.sql.functions import sum, count, when, col, lit, concat, round, min, coalesce

i = innings.alias("i")
t = team.alias("t")
s = score_by_ball.alias("s")
p = player.alias("p")

i.join(
    t,
    i.bowling_team_id == t.team_id,
    "inner"
).join(
    s,
    (s.match_id == i.match_id) &
    (s.innings_no == i.innings_no),
    "inner"
).join(
    p,
    s.bowler_id == p.player_id,
    "inner"
).groupBy(
    i.match_id,
    i.innings_no,
    t.team_name.alias("bowling_team_name"),
    p.player_name.alias("bowler_name")
).agg(
    # overs
    when(
        ((count("*") - count(s.wides) - count(s.noballs)) % 6) == 0,
        ((count("*") - count(s.wides) - count(s.noballs)) / 6)
        .cast("int")
        .cast("string")
    ).otherwise(
        concat(
            (((count("*") - count(s.wides) - count(s.noballs)) / 6)
             .cast("int")),
            lit("."),
            (((count("*") - count(s.wides) - count(s.noballs)) % 6)
             .cast("int"))
        )
    ).alias("total_overs"),
    # runs conceded
    sum(
        s.runs_off_bat +
        when(s.wides.isNull(), 0).otherwise(s.wides) +
        coalesce(s.noballs, lit(0))
    ).alias("total_runs"),
    # wickets
    count(s.wicket_type).alias("total_wickets"),
    # economy
    round(
        sum(
            s.runs_off_bat +
            coalesce(s.wides, lit(0)) +
            coalesce(s.noballs, lit(0))
        ) /
        (
            (count("*") - count(s.wides) - count(s.noballs)) / 6
        ),
        2
    ).alias("economy"),
    # dot balls
    sum(
        when(
            (s.runs_off_bat == 0) &
            s.wides.isNull() &
            s.noballs.isNull(),
            1
        ).otherwise(0)
    ).alias("dots"),
).orderBy(
    "match_id",
    "innings_no",
    min(s.ball_no)
).display()

#### Output -
![image_1781587194695.png](./image_1781587194695.png "image_1781587194695.png")